# LangGraph Liver/Tumor Segmentation Inference Pipeline

This notebook runs a per-case LangGraph pipeline:

1. Prepare A/P/D input files
2. Resample and register A/D to portal phase
3. Extract liver masks and compute AP/AD/PD liver Dice
4. Retry registration up to 5 attempts until mean liver Dice passes the threshold
5. If registration succeeds, run independent tumor inference for UNet, SwinUNETR, 3DSAM-adapter, and nnUNetv2
6. Save every attempt, tumor prediction, metrics, and summary under `Results/{case_id}/`

Notes:
- `langgraph/files/registration.py` and `resampler.py` are used as provided. Only output path naming was adjusted to `registration/attempt_n`.
- `langgraph/files/liver_extractor.py` currently appears to duplicate `resampler.py`. Until a real `LiverExtractor` class is provided there, this notebook uses the existing liver extraction function from `externalregis.py` as a fallback.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Dict, List, Optional, TypedDict

import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
try:
    from langgraph.graph import END, START, StateGraph
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "LangGraph is required for this notebook. Install it in this environment with: pip install langgraph"
    ) from exc

from config import DATA_PATH1, DATA_PATH2
from main import format_path_part, get_model
from inference import load_checkpoint, load_case, save_nifti, sliding_window_predict
from langgraph.files.config import REGISTRATION_CONFIGS
from langgraph.files.resampler import Resampler
from langgraph.files.registration import Registration

try:
    from externalregis import extract_segmentation as fallback_extract_liver
except Exception as exc:
    fallback_extract_liver = None
    print(f"externalregis liver extraction fallback unavailable: {exc}")


In [ ]:
@dataclass
class PipelineConfig:
    excel_path: str = "./alldata_metrics.xlsx"
    data_root1: str = DATA_PATH1
    data_root2: str = DATA_PATH2
    results_root: str = "./Results"
    test_size: int = 34
    liver_dice_threshold: float = 0.95
    max_registration_attempts: int = 5
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Direct model inference settings
    window: tuple[float, float] = (-200, 300)
    roi_size: tuple[int, int, int] = (96, 128, 128)
    sw_batch_size: int = 4
    sw_overlap: float = 0.5
    sw_mode: str = "gaussian"
    threshold: float = 0.5

    # Model architecture args must match training.
    channels: tuple[int, ...] = (16, 32, 64, 128, 256)
    strides: tuple[int, ...] = (2, 2, 2, 2)
    num_res_units: int = 2
    img_size: tuple[int, int, int] = (96, 384, 384)
    feature_size: int = 48
    use_checkpoint: bool = False
    sam_patch_size: int = 16
    sam_tubelet_size: int = 16
    sam_embed_dim: int = 128
    sam_depth: int = 4
    sam_num_heads: int = 4
    sam_encoder_channels: int = 128
    sam_decoder_channels: int = 64
    sam_adapter_ratio: float = 0.5

    checkpoints: Dict[str, str] = None

    # nnUNetv2 settings
    nnunet_input_dir: str = "./nnUNet/nnUNet_raw/Dataset001/imagesTs"
    nnunet_dataset: str = "001"
    nnunet_config: str = "3d_fullres"
    nnunet_save_probabilities: bool = True
    nnunet_raw: Optional[str] = None
    nnunet_preprocessed: Optional[str] = None
    nnunet_results: Optional[str] = None

    def __post_init__(self):
        if self.checkpoints is None:
            self.checkpoints = {
                "unet": "./checkpoints/unet/best_model.pt",
                "swinunetr": "./checkpoints/swinunetr/best_model.pt",
                "sam3d_adapter": "./checkpoints/sam_adapter/best_model.pt",
            }

CFG = PipelineConfig()
Path(CFG.results_root).mkdir(parents=True, exist_ok=True)
asdict(CFG)


In [ ]:
def load_test_dataframe(cfg: PipelineConfig) -> pd.DataFrame:
    data = pd.read_excel(cfg.excel_path)
    data = data.sort_values("tumor_size").reset_index(drop=True)
    return data.iloc[:cfg.test_size].copy()


def get_case_from_row(row: pd.Series, cfg: PipelineConfig) -> Dict[str, Any]:
    subject = format_path_part(row["subject"])
    date = format_path_part(row["date"])
    if date == "0":
        source_dir = Path(cfg.data_root2) / subject
        case_id = subject
    else:
        source_dir = Path(cfg.data_root1) / subject / date
        case_id = f"{subject}_{date}"

    return {
        "subject": subject,
        "date": date,
        "case_id": case_id,
        "source_dir": str(source_dir),
        "A": str(source_dir / "A.nii.gz"),
        "P": str(source_dir / "P.nii.gz"),
        "D": str(source_dir / "D.nii.gz"),
        "label": str(source_dir / "label.nii.gz"),
    }


def get_test_cases(cfg: PipelineConfig) -> List[Dict[str, Any]]:
    df = load_test_dataframe(cfg)
    cases = []
    for _, row in df.iterrows():
        case = get_case_from_row(row, cfg)
        required = [case["A"], case["P"], case["D"], case["label"]]
        if all(Path(p).exists() for p in required):
            cases.append(case)
        else:
            print(f"skip missing case: {case['case_id']}")
    return cases


def write_json(path: str | Path, data: Dict[str, Any]):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def copy_phase_inputs(case: Dict[str, Any], case_result_dir: Path) -> Path:
    input_dir = case_result_dir / "input"
    input_dir.mkdir(parents=True, exist_ok=True)
    for phase in ("A", "P", "D"):
        shutil.copy2(case[phase], input_dir / f"{phase}.nii.gz")
    return input_dir


In [ ]:
def binary_array(path: str | Path) -> np.ndarray:
    return sitk.GetArrayFromImage(sitk.ReadImage(str(path))) > 0


def dice_score(pred: np.ndarray, gt: np.ndarray, eps: float = 1e-8) -> float:
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return float((2 * inter + eps) / (denom + eps))


def iou_score(pred: np.ndarray, gt: np.ndarray, eps: float = 1e-8) -> float:
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return float((inter + eps) / (union + eps))


def resample_label_to_reference(label_path: str | Path, reference_path: str | Path, output_path: str | Path) -> str:
    label_img = sitk.ReadImage(str(label_path))
    ref_img = sitk.ReadImage(str(reference_path))
    resampled = sitk.Resample(
        label_img,
        ref_img,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        label_img.GetPixelID(),
    )
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sitk.WriteImage(resampled, str(output_path))
    return str(output_path)


def compute_prediction_metrics(pred_path: str | Path, gt_path: str | Path) -> Dict[str, float]:
    pred = binary_array(pred_path)
    gt = binary_array(gt_path)
    return {"dice": dice_score(pred, gt), "iou": iou_score(pred, gt)}


In [ ]:
def extract_liver_masks(attempt_dir: Path) -> Dict[str, str]:
    # Preferred future integration point: langgraph/files/liver_extractor.py should provide LiverExtractor.
    try:
        from langgraph.files.liver_extractor import LiverExtractor  # type: ignore
        extractor = LiverExtractor(input_folder=str(attempt_dir), output_path=str(attempt_dir))
        result = extractor.run()
        if isinstance(result, dict):
            return result
    except Exception:
        pass

    if fallback_extract_liver is None:
        raise RuntimeError(
            "No usable LiverExtractor found. langgraph/files/liver_extractor.py currently does not define "
            "LiverExtractor, and externalregis.extract_segmentation fallback is unavailable."
        )

    liver_root = attempt_dir / "liver_masks"
    mask_paths = {}
    for phase in ("A", "P", "D"):
        phase_out = liver_root / phase
        phase_out.mkdir(parents=True, exist_ok=True)
        fallback_extract_liver(str(attempt_dir / f"{phase}.nii.gz"), str(phase_out))
        mask_paths[phase] = str(phase_out / "liver.nii.gz")

    # Requirement asks for attempt_dir/liver.nii.gz. Portal liver mask is used as the representative mask.
    shutil.copy2(mask_paths["P"], attempt_dir / "liver.nii.gz")
    return mask_paths


def compute_liver_dice(mask_paths: Dict[str, str]) -> Dict[str, float]:
    liver_a = binary_array(mask_paths["A"])
    liver_p = binary_array(mask_paths["P"])
    liver_d = binary_array(mask_paths["D"])
    ap = dice_score(liver_a, liver_p)
    ad = dice_score(liver_a, liver_d)
    pd_ = dice_score(liver_p, liver_d)
    return {"APdice": ap, "ADdice": ad, "PDdice": pd_, "MeanDice": float((ap + ad + pd_) / 3)}


In [ ]:
class PipelineState(TypedDict, total=False):
    cfg: PipelineConfig
    case: Dict[str, Any]
    case_result_dir: str
    input_dir: str
    attempts: List[Dict[str, Any]]
    registration_success: bool
    final_attempt_dir: Optional[str]
    final_liver_dice: Optional[float]
    tumor_metrics: Dict[str, Dict[str, float]]
    failed_reason: Optional[str]
    summary_path: Optional[str]


def prepare_case_node(state: PipelineState) -> PipelineState:
    cfg = state["cfg"]
    case = state["case"]
    case_result_dir = Path(cfg.results_root) / case["case_id"]
    case_result_dir.mkdir(parents=True, exist_ok=True)
    input_dir = copy_phase_inputs(case, case_result_dir)
    write_json(case_result_dir / "case.json", case)
    state.update({
        "case_result_dir": str(case_result_dir),
        "input_dir": str(input_dir),
        "attempts": [],
        "registration_success": False,
        "final_attempt_dir": None,
        "final_liver_dice": None,
        "tumor_metrics": {},
        "failed_reason": None,
    })
    return state


def registration_loop_node(state: PipelineState) -> PipelineState:
    cfg = state["cfg"]
    input_dir = Path(state["input_dir"])
    case_result_dir = Path(state["case_result_dir"])

    Resampler(input_folder=str(input_dir), output_path=str(case_result_dir)).run()

    attempts = []
    for attempt in range(1, cfg.max_registration_attempts + 1):
        reg_param = REGISTRATION_CONFIGS[attempt]
        attempt_dir = Registration(
            reg_param=reg_param,
            input_folder=str(case_result_dir),
            output_path=str(case_result_dir),
            attempt=attempt - 1,
        ).run()
        attempt_dir = Path(attempt_dir)

        mask_paths = extract_liver_masks(attempt_dir)
        dice_info = compute_liver_dice(mask_paths)
        dice_info["attempt"] = attempt
        dice_info["threshold"] = cfg.liver_dice_threshold
        write_json(attempt_dir / "liver_dice.json", dice_info)

        attempt_record = {
            "attempt": attempt,
            "attempt_dir": str(attempt_dir),
            "liver_dice": dice_info,
            "config": reg_param.get("name", f"Attempt {attempt}"),
        }
        attempts.append(attempt_record)

        if dice_info["MeanDice"] >= cfg.liver_dice_threshold:
            state["registration_success"] = True
            state["final_attempt_dir"] = str(attempt_dir)
            state["final_liver_dice"] = dice_info["MeanDice"]
            break

    state["attempts"] = attempts
    if not state.get("registration_success"):
        state["failed_reason"] = "registration_liver_dice_below_threshold"
        if attempts:
            state["final_attempt_dir"] = attempts[-1]["attempt_dir"]
            state["final_liver_dice"] = attempts[-1]["liver_dice"]["MeanDice"]
    return state


def route_after_registration(state: PipelineState) -> str:
    return "tumor" if state.get("registration_success") else "finalize"


In [ ]:
def make_model_args(cfg: PipelineConfig, model_name: str) -> SimpleNamespace:
    internal_name = "sam_adapter" if model_name == "sam3d_adapter" else model_name
    return SimpleNamespace(
        model=internal_name,
        checkpoint=cfg.checkpoints.get(model_name),
        window=list(cfg.window),
        roi_size=list(cfg.roi_size),
        sw_batch_size=cfg.sw_batch_size,
        sw_overlap=cfg.sw_overlap,
        sw_mode=cfg.sw_mode,
        use_sliding_window=True,
        threshold=cfg.threshold,
        device=cfg.device,
        channels=list(cfg.channels),
        strides=list(cfg.strides),
        num_res_units=cfg.num_res_units,
        img_size=list(cfg.img_size),
        feature_size=cfg.feature_size,
        use_checkpoint=cfg.use_checkpoint,
        sam_patch_size=cfg.sam_patch_size,
        sam_tubelet_size=cfg.sam_tubelet_size,
        sam_embed_dim=cfg.sam_embed_dim,
        sam_depth=cfg.sam_depth,
        sam_num_heads=cfg.sam_num_heads,
        sam_encoder_channels=cfg.sam_encoder_channels,
        sam_decoder_channels=cfg.sam_decoder_channels,
        sam_adapter_ratio=cfg.sam_adapter_ratio,
    )


@torch.no_grad()
def run_direct_model(model_name: str, attempt_dir: Path, output_dir: Path, cfg: PipelineConfig) -> Optional[str]:
    args = make_model_args(cfg, model_name)
    checkpoint = args.checkpoint
    if checkpoint is None or not Path(checkpoint).exists():
        print(f"skip {model_name}: checkpoint not found -> {checkpoint}")
        return None

    device = torch.device(cfg.device)
    model = get_model(args).to(device)
    model.eval()
    load_checkpoint(model, checkpoint, device)

    image, reference_image = load_case(
        str(attempt_dir / "A.nii.gz"),
        str(attempt_dir / "P.nii.gz"),
        str(attempt_dir / "D.nii.gz"),
        args.window,
    )
    image = image.to(device)
    logits = sliding_window_predict(
        model,
        image,
        args.roi_size,
        args.sw_batch_size,
        args.sw_overlap,
        args.sw_mode,
    )
    probability = torch.sigmoid(logits)[0, 0].cpu().numpy()
    mask = (probability >= cfg.threshold).astype(np.uint8)

    output_dir.mkdir(parents=True, exist_ok=True)
    pred_path = output_dir / "pred.nii.gz"
    save_nifti(mask, reference_image, str(pred_path))
    return str(pred_path)


def run_nnunetv2(attempt_dir: Path, output_dir: Path, cfg: PipelineConfig) -> Optional[str]:
    input_dir = output_dir / "input"
    input_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(attempt_dir / "A.nii.gz", input_dir / "case_0000.nii.gz")
    shutil.copy2(attempt_dir / "P.nii.gz", input_dir / "case_0001.nii.gz")
    shutil.copy2(attempt_dir / "D.nii.gz", input_dir / "case_0002.nii.gz")
    output_dir.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    if cfg.nnunet_raw:
        env["nnUNet_raw"] = cfg.nnunet_raw
    if cfg.nnunet_preprocessed:
        env["nnUNet_preprocessed"] = cfg.nnunet_preprocessed
    if cfg.nnunet_results:
        env["nnUNet_results"] = cfg.nnunet_results

    cmd = [
        "nnUNetv2_predict",
        "-i", str(input_dir),
        "-o", str(output_dir),
        "-d", cfg.nnunet_dataset,
        "-c", cfg.nnunet_config,
    ]
    if cfg.nnunet_save_probabilities:
        cmd.append("--save_probabilities")

    log_path = output_dir / "nnunetv2_predict.log"
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.run(cmd, stdout=log_file, stderr=subprocess.STDOUT, text=True, env=env)

    pred_candidates = sorted(output_dir.glob("*.nii.gz"))
    if process.returncode != 0 or not pred_candidates:
        print(f"nnUNetv2 failed. See {log_path}")
        return None

    pred_path = output_dir / "pred.nii.gz"
    shutil.copy2(pred_candidates[0], pred_path)
    return str(pred_path)


def tumor_extraction_node(state: PipelineState) -> PipelineState:
    cfg = state["cfg"]
    case = state["case"]
    attempt_dir = Path(state["final_attempt_dir"])
    tumor_root = Path(state["case_result_dir"]) / "tumor"
    gt_path = resample_label_to_reference(case["label"], attempt_dir / "P.nii.gz", Path(state["case_result_dir"]) / "gt_label_resampled.nii.gz")

    model_outputs = {
        "unet": lambda out: run_direct_model("unet", attempt_dir, out, cfg),
        "swinunetr": lambda out: run_direct_model("swinunetr", attempt_dir, out, cfg),
        "sam3d_adapter": lambda out: run_direct_model("sam3d_adapter", attempt_dir, out, cfg),
        "nnunetv2": lambda out: run_nnunetv2(attempt_dir, out, cfg),
    }

    tumor_metrics = {}
    for model_name, runner in model_outputs.items():
        model_dir = tumor_root / model_name
        pred_path = runner(model_dir)
        if pred_path is None:
            tumor_metrics[model_name] = {"status": "failed_or_skipped"}
            write_json(model_dir / "metrics.json", tumor_metrics[model_name])
            continue

        metrics = compute_prediction_metrics(pred_path, gt_path)
        metrics["status"] = "ok"
        tumor_metrics[model_name] = metrics
        write_json(model_dir / "metrics.json", metrics)

    state["tumor_metrics"] = tumor_metrics
    return state


In [ ]:
def finalize_node(state: PipelineState) -> PipelineState:
    case_result_dir = Path(state["case_result_dir"])
    summary = {
        "case": state["case"],
        "registration_success": state.get("registration_success", False),
        "failed_reason": state.get("failed_reason"),
        "registration_attempts": len(state.get("attempts", [])),
        "final_attempt_dir": state.get("final_attempt_dir"),
        "final_liver_dice": state.get("final_liver_dice"),
        "attempts": state.get("attempts", []),
        "tumor_metrics": state.get("tumor_metrics", {}),
    }
    summary_path = case_result_dir / "summary.json"
    write_json(summary_path, summary)
    state["summary_path"] = str(summary_path)
    return state


def build_graph():
    graph = StateGraph(PipelineState)
    graph.add_node("prepare", prepare_case_node)
    graph.add_node("registration", registration_loop_node)
    graph.add_node("tumor", tumor_extraction_node)
    graph.add_node("finalize", finalize_node)

    graph.add_edge(START, "prepare")
    graph.add_edge("prepare", "registration")
    graph.add_conditional_edges(
        "registration",
        route_after_registration,
        {"tumor": "tumor", "finalize": "finalize"},
    )
    graph.add_edge("tumor", "finalize")
    graph.add_edge("finalize", END)
    return graph.compile()

pipeline_graph = build_graph()
pipeline_graph


## Single Mode

Set `SINGLE_CASE_ID` to a case id such as `7088634_20170515` or `1043712`. If it is `None`, the first available test case is used.


In [ ]:
SINGLE_CASE_ID = None  # example: "7088634_20170515" or "1043712"

test_cases = get_test_cases(CFG)
if SINGLE_CASE_ID is None:
    single_case = test_cases[0]
else:
    single_case = next(case for case in test_cases if case["case_id"] == SINGLE_CASE_ID)

single_result = pipeline_graph.invoke({"cfg": CFG, "case": single_case})
single_result["summary_path"]


## Batch Mode

Set `RUN_BATCH = True` to process all test cases. Results are saved independently under `Results/{case_id}/`.


In [ ]:
RUN_BATCH = False

batch_summaries = []
if RUN_BATCH:
    for case in get_test_cases(CFG):
        print(f"\n=== Running {case['case_id']} ===")
        result = pipeline_graph.invoke({"cfg": CFG, "case": case})
        batch_summaries.append(result["summary_path"])

    write_json(Path(CFG.results_root) / "batch_summary.json", {"summaries": batch_summaries})

batch_summaries[:5]
